StableBaselines3 based SAC model for Rover2026 training, CUDA-accelerated, checkpoint-and-saving system

Loads environment but headless, for no render training.

In [10]:
import os
os.environ["MUJOCO_GL"] = "egl"  # Headless GPU rendering

imports below

In [2]:
import time
import torch
from gym_wrapper import RobosuiteGymWrapper
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import CheckpointCallback

[robosuite WARNING] No private macro file found! (macros.py:57)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:58)
[robosuite WARNING] To setup, run: python /LearnFlake/src/external_pkgs/RoboSuite/robosuite/scripts/setup_macros.py (macros.py:59)
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
/usr/local/lib/python3.10/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pi

Constants defined for model, evaluation, checkpoints and training parameters.

In [3]:
MODEL_PATH = "sac_Rover2026_V1_model"
SAVE_NAME = "sac_Rover2026_V1_model"
TOTAL_TIMESTEPS = 1_000
FAST_MODE = True  # Flip to False for full logging/eval/checkpoints
CHECKPOINT_FREQ = 0 if FAST_MODE else 100  # Saves 10 checkpoints throughout training
N_EVAL_EPISODES = 2 if FAST_MODE else 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROBOT = "Rover2026"

Environment defined using `RobosuiteGymWrapper`

In [4]:
env = RobosuiteGymWrapper(
    "Lift",
    robots=ROBOT,
    has_renderer=False,
    has_offscreen_renderer=False,
    use_camera_obs=False,
    reward_shaping=True
)

[robosuite INFO] Loading controller configuration from: /LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/robots/default_rover2026.json (composite_controller_factory.py:121)
[robosuite INFO] Loading controller configuration from: /LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/robots/default_rover2026.json (composite_controller_factory.py:121)
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Define model and loads a new one if the path doesn't exist yet:

In [5]:
if os.path.exists(MODEL_PATH):
    model = SAC.load(MODEL_PATH, env=env, device=DEVICE)
    print(f"Loaded model with {model.num_timesteps} timesteps")
else:
    model = SAC(
        "MlpPolicy",
        env,
        verbose=1,
        device=DEVICE,
        tensorboard_log="./logs/"
    )
    print("Created new model")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Created new model


Checkpoints created. When training a million times, it was a pain to not be able to see progress hence this exists

In [6]:
checkpoint_callback = None
if CHECKPOINT_FREQ and CHECKPOINT_FREQ > 0:
    checkpoint_callback = CheckpointCallback(
        save_freq=CHECKPOINT_FREQ,
        save_path="./checkpoints/",
        name_prefix="sac_rover"
    )

Training below

In [7]:
start_time = time.time()
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    reset_num_timesteps=False,
    callback=checkpoint_callback
)
elapsed = time.time() - start_time
print(f"Training took {elapsed:.2f} seconds")

[robosuite INFO] Loading controller configuration from: /LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/robots/default_rover2026.json (composite_controller_factory.py:121)


Logging to ./logs/SAC_0


[robosuite INFO] Loading controller configuration from: /LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/robots/default_rover2026.json (composite_controller_factory.py:121)


Training took 40.97 seconds


Evaluation below

In [8]:
start_time2 = time.time()
if N_EVAL_EPISODES and N_EVAL_EPISODES > 0:
    mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=N_EVAL_EPISODES)
    print(f"Mean reward: {mean_reward:.2f}, Std reward: {std_reward:.2f}")
    elapsed2 = time.time() - start_time2
    print(f"Evaluation took {elapsed2:.2f} seconds")

/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
[robosuite INFO] Loading controller configuration from: /LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/robots/default_rover2026.json (composite_controller_factory.py:121)
[robosuite INFO] Loading controller configuration from: /LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/robots/default_rover2026.json (composite_controller_factory.py:121)
[robosuite INFO] Loading controller configuration from: /LearnFlake/src/external_pkgs/RoboSuite/robosuite/controllers/config/robots/default_rover2026.json (composite_controller_factory.py:121)


Mean reward: 0.21, Std reward: 0.19
Evaluation took 30.33 seconds


Model saved to path

In [9]:
model.save(SAVE_NAME)
print(f"Model saved to {SAVE_NAME}.zip")

env.close()

Model saved to sac_Rover2026_V1_model.zip
